In [1]:
import cvxpy as cp
import numpy as np

# ==========================================
# Question 2
# ==========================================


# ==========================================
# 1. PARAMÉTRAGE ET DONNÉES
# ==========================================
print("Initialisation des données pour Q2...")

bureaux = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3', 'E1', 'E2']
services = ['P', 'S', 'O', 'T', 'M']
nb_bur = len(bureaux)
nb_serv = len(services)
nb_trans = 5

b_idx = {b: i for i, b in enumerate(bureaux)}
s_idx = {s: i for i, s in enumerate(services)}

# --- États Initiaux et Finaux ---
init_state = np.zeros((nb_serv, nb_bur))
config_init = {
    'A1': ['P', 'P'], 'A2': ['P', 'P'], 'B1': ['O', 'M'], 'B2': ['S', 'S'], 'B3': ['O', 'T'],
    'C1': ['T', 'T'], 'C2': ['S', 'S'], 'D1': ['T'], 'D2': ['M'], 'D3': ['M', 'M']
}
for b, servs in config_init.items():
    for s in servs: init_state[s_idx[s], b_idx[b]] += 1

final_state = np.zeros((nb_serv, nb_bur))
config_final = {
    'A1': ['P', 'P'], 'A2': ['P', 'P'], 'B1': ['M'], 'B2': ['M', 'M'], 'B3': ['M'],
    'C1': ['S', 'S'], 'C2': ['S', 'S'], 'D1': ['T'], 'D2': ['T', 'T'], 'D3': ['T'],
    'E1': ['O'], 'E2': ['O']
}
for b, servs in config_final.items():
    for s in servs: final_state[s_idx[s], b_idx[b]] += 1

# Rénovations
renovations = {
    0: ['B1', 'B2', 'B3'],  # Phase 1 fermée
    1: ['D1', 'D2', 'D3'],  # Phase 2 fermée
    2: ['C1', 'C2'],        # Phase 3 fermée
    3: ['A1', 'A2'],        # Phase 4 fermée
    4: []                   # Phase 5 (tout ouvert)
}

# ==========================================
# 2. MODÉLISATION LP
# ==========================================
print("Construction du modèle LP...")

x = cp.Variable((nb_serv, nb_trans, nb_bur, nb_bur), nonneg=True)

constraints = []
objective_terms = []

# Pour stocker les références aux contraintes de capacité (pour l'analyse duale)
capacity_constrs = {}

for p in range(nb_trans):
    closed_offices = renovations[p]
    closed_indices = [b_idx[b] for b in closed_offices]

    for j in range(nb_bur):
        # 1. Objectif : Minimiser mouvements (i != j)
        flux_mouv = x[:, p, [i for i in range(nb_bur) if i != j], j]
        objective_terms.append(cp.sum(flux_mouv))

        # 2. Capacité (<= 2)
        # On garde la référence de la contrainte dans un dico pour l'analyse duale
        flux_entrant_total = cp.sum(x[:, p, :, j])
        constr = (flux_entrant_total <= 2)
        constraints.append(constr)
        capacity_constrs[(p, j)] = constr # Clé : (phase, bureau)

        # 3. Rénovation (Fermeture)
        if j in closed_indices:
            constraints.append(flux_entrant_total == 0)

# 4. Conservation de flux
for s in range(nb_serv):
    # Départ initial
    for i in range(nb_bur):
        constraints.append(cp.sum(x[s, 0, i, :]) == init_state[s, i])
    # Intermédiaire
    for p in range(nb_trans - 1):
        for k in range(nb_bur):
            constraints.append(cp.sum(x[s, p, :, k]) == cp.sum(x[s, p+1, k, :]))
    # Arrivée finale
    for j in range(nb_bur):
        constraints.append(cp.sum(x[s, nb_trans-1, :, j]) == final_state[s, j])

# ==========================================
# 3. RÉSOLUTION
# ==========================================
print("Résolution en cours...")
prob = cp.Problem(cp.Minimize(cp.sum(objective_terms)), constraints)
prob.solve()

print(f"Status : {prob.status}")

if prob.status == 'optimal' or prob.status == 'optimal_inaccurate':
    print(f"\n{'='*60}")
    print(f"RÉSULTAT QUESTION 2 : {prob.value:.2f} Mouvements")
    print(f"{'='*60}")

    # --- ANALYSE DUALE (SHADOW PRICES) ---
    print("\n--- ANALYSE DUALE (Shadow Prices > 0.001) ---")
    print("Signification : Gain potentiel si on augmentait la capacité de ce bureau.")
    for (p, j), constr in capacity_constrs.items():
        dual_val = constr.dual_value
        # On vérifie si la valeur duale est significative
        if dual_val is not None and dual_val > 0.001:
            print(f"  Phase {p} - Bureau {bureaux[j]:<3} : λ = {dual_val:.4f}")

    # --- AFFICHAGE DÉTAILLÉ ---
    print(f"\n{'='*60}")
    print("DÉTAIL DE LA SOLUTION (PHASE PAR PHASE)")
    print(f"{'='*60}")

    x_val = x.value
    current_occupancy = init_state.copy()

    for p in range(nb_trans + 1):
        print(f"\n--- PHASE {p} ---", end="")
        if p > 0 and p <= 5:
            closed = renovations[p-1]
            if closed: print(f" (Travaux: {closed})")
        else: print("")

        for j in range(nb_bur):
            occupants = []
            for s in range(nb_serv):
                val = current_occupancy[s, j]
                if val > 0.01:
                    occupants.append(f"{services[s]}:{val:.2f}")
            if occupants:
                print(f"  Bureau {bureaux[j]:<3} : {', '.join(occupants)}")

        # Mise à jour pour la phase suivante
        if p < nb_trans:
            next_occupancy = np.zeros((nb_serv, nb_bur))
            for s in range(nb_serv):
                for j in range(nb_bur):
                    flux_in = np.sum(x_val[s, p, :, j])
                    next_occupancy[s, j] = flux_in
            current_occupancy = next_occupancy

else:
    print("Erreur de résolution.")

Initialisation des données pour Q2...
Construction du modèle LP...
Résolution en cours...


C:\Users\Magomed Tsitsiev\anaconda3\Lib\site-packages\cvxpy\reductions\solvers\solving_chain_utils.py:41: UserWarning: The problem has an expression with dimension greater than 2. Defaulting to the SCIPY backend for canonicalization.
  warnings.warn(UserWarning(


Status : optimal

RÉSULTAT QUESTION 2 : 30.00 Mouvements

--- ANALYSE DUALE (Shadow Prices > 0.001) ---
Signification : Gain potentiel si on augmentait la capacité de ce bureau.
  Phase 0 - Bureau A1  : λ = 1.7406
  Phase 0 - Bureau A2  : λ = 1.7406
  Phase 0 - Bureau C1  : λ = 1.8188
  Phase 0 - Bureau C2  : λ = 1.7406
  Phase 0 - Bureau D1  : λ = 0.7406
  Phase 0 - Bureau D2  : λ = 0.7406
  Phase 0 - Bureau D3  : λ = 1.5191
  Phase 0 - Bureau E1  : λ = 1.2737
  Phase 0 - Bureau E2  : λ = 1.2737
  Phase 1 - Bureau A1  : λ = 1.1162
  Phase 1 - Bureau A2  : λ = 1.1162
  Phase 1 - Bureau B1  : λ = 1.1162
  Phase 1 - Bureau B2  : λ = 1.3051
  Phase 1 - Bureau B3  : λ = 1.1162
  Phase 1 - Bureau C1  : λ = 1.5576
  Phase 1 - Bureau C2  : λ = 1.1162
  Phase 1 - Bureau E1  : λ = 1.5830
  Phase 1 - Bureau E2  : λ = 1.5830
  Phase 2 - Bureau B2  : λ = 0.2166
  Phase 3 - Bureau B2  : λ = 0.2480
  Phase 4 - Bureau A1  : λ = 1.2635
  Phase 4 - Bureau A2  : λ = 1.2635
  Phase 4 - Bureau B2  : λ = 1

In [2]:

# ==========================================
# 1. Q3 - Version LP
# ==========================================


# ==========================================
# 1. DONNÉES ET PARAMÈTRES
# ==========================================
print("Initialisation des données...")

bureaux = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3', 'E1', 'E2']
services = ['P', 'S', 'O', 'T', 'M']
# P=Presidence, S=Students, O=Opti, T=TCS, M=Maths

nb_bur = len(bureaux)
nb_serv = len(services)
nb_trans = 5  # 5 transitions (0->1, 1->2, 2->3, 3->4, 4->5)

b_idx = {b: i for i, b in enumerate(bureaux)}
s_idx = {s: i for i, s in enumerate(services)}

# --- État Initial (Phase 0) ---
init_state = np.zeros((nb_serv, nb_bur))
init_config = {
    'A1': ['P', 'P'], 'A2': ['P', 'P'],
    'B1': ['O', 'M'], 'B2': ['S', 'S'], 'B3': ['O', 'T'],
    'C1': ['T', 'T'], 'C2': ['S', 'S'],
    'D1': ['T'], 'D2': ['M'], 'D3': ['M', 'M']
    # E est vide
}
for bureau, servs in init_config.items():
    for s in servs:
        init_state[s_idx[s], b_idx[bureau]] += 1

# --- État Final Cible (Phase 5) ---
final_state = np.zeros((nb_serv, nb_bur))
final_config = {
    'A1': ['P', 'P'], 'A2': ['P', 'P'],
    'B1': ['M'], 'B2': ['M', 'M'], 'B3': ['M'],
    'C1': ['S', 'S'], 'C2': ['S', 'S'],
    'D1': ['T'], 'D2': ['T', 'T'], 'D3': ['T'],
    'E1': ['O'], 'E2': ['O']
}
for bureau, servs in final_config.items():
    for s in servs:
        final_state[s_idx[s], b_idx[bureau]] += 1

# --- Rénovations (Bureaux fermés APRES la transition p) ---
# renovations[0] -> Phase 1, renovations[1] -> Phase 2, etc.
renovations = [
    ['B1', 'B2', 'B3'],  # Phase 1 : B fermé
    ['D1', 'D2', 'D3'],  # Phase 2 : D fermé
    ['C1', 'C2'],        # Phase 3 : C fermé
    ['A1', 'A2'],        # Phase 4 : A fermé
    []                   # Phase 5 : Tout ouvert (Etat final)
]

# --- Graphe de Voisinage ---
edges = [
    ('A1', 'D3'), ('D3', 'D2'), ('D2', 'D1'), ('D1', 'C2'),
    ('C2', 'C1'), ('C1', 'B3'), ('B3', 'B2'), ('B2', 'B1'),
    ('B1', 'A2'), ('A2', 'A1'),
    ('D2', 'E1'), ('E1', 'E2'), ('E2', 'B2')
]

idx_P = s_idx['P']
idx_S = s_idx['S']

# ==========================================
# 2. MODÉLISATION (LP)
# ==========================================
print("Construction du modèle LP (Relaxation Stricte)...")

# x[s, p, i, j] : Quantité de service 's' bougeant de 'i' vers 'j' à la transition 'p'
x = cp.Variable((nb_serv, nb_trans, nb_bur, nb_bur), nonneg=True)

constraints = []
objective_terms = []

# Pour suivre l'occupation à chaque phase : occ[p][s, j]
# On va stocker les EXPRESSIONS CVXPY pour les contraintes
# Et on recalculera les VALEURS numériques après résolution pour l'affichage
occ_expr = []

# --- Phase 0 (Initiale) ---
# C'est une constante, mais on la met dans la liste pour uniformiser
occ_0 = {}
for s in range(nb_serv):
    for j in range(nb_bur):
        occ_0[(s, j)] = init_state[s, j]
occ_expr.append(occ_0)

# --- Boucle sur les Transitions ---
for p in range(nb_trans):
    # L'occupation à la phase p+1 dépend de la phase p et des flux x[p]
    occ_next = {}

    # Bureaux fermés à la phase p+1
    closed_offices = renovations[p]

    for j in range(nb_bur):
        # 1. OBJECTIF : Minimiser les mouvements (flux i -> j avec i != j)
        flux_mouv = x[:, p, [i for i in range(nb_bur) if i != j], j]
        objective_terms.append(cp.sum(flux_mouv))

        # Calcul de l'occupation future pour chaque service
        for s in range(nb_serv):
            # Flux net entrant en j : Somme(x[s, p, :, j])
            # Note : La conservation de flux impose que Somme(x[s, p, j, :]) = occ_prev[s, j]
            # Donc l'occupation suivante est simplement la somme des flux entrants
            occ_next[(s, j)] = cp.sum(x[s, p, :, j])

            # Contrainte de conservation (Flux sortant = Occupation précédente)
            constraints.append(cp.sum(x[s, p, j, :]) == occ_expr[p][(s, j)])

        # 2. CAPACITÉ : Max 2 personnes (somme des services)
        total_occ_j = cp.sum([occ_next[(s, j)] for s in range(nb_serv)])
        constraints.append(total_occ_j <= 2.0)

        # 3. RÉNOVATIONS
        if bureaux[j] in closed_offices:
            constraints.append(total_occ_j == 0)

        # 4. EXCLUSION STRICTE (COHABITATION)
        # Normalisation : (P/2 + S/2) <= 1
        val_P = occ_next[(idx_P, j)]
        val_S = occ_next[(idx_S, j)]
        constraints.append(val_P/2.0 + val_S/2.0 <= 1.0)

    occ_expr.append(occ_next)

    # 5. EXCLUSION STRICTE (VOISINAGE) pour la phase p+1
    for (u_name, v_name) in edges:
        u, v = b_idx[u_name], b_idx[v_name]
        val_P_u = occ_next[(idx_P, u)]
        val_S_v = occ_next[(idx_S, v)]
        val_S_u = occ_next[(idx_S, u)]
        val_P_v = occ_next[(idx_P, v)]

        constraints.append(val_P_u/2.0 + val_S_v/2.0 <= 1.0)
        constraints.append(val_S_u/2.0 + val_P_v/2.0 <= 1.0)

# --- Contrainte Finale ---
# La dernière occupation (Phase 5) doit être égale à l'état final cible
for s in range(nb_serv):
    for j in range(nb_bur):
        constraints.append(occ_expr[nb_trans][(s, j)] == final_state[s, j])

# ==========================================
# 3. RÉSOLUTION
# ==========================================
print("Résolution en cours...")
prob = cp.Problem(cp.Minimize(cp.sum(objective_terms)), constraints)

try:
    prob.solve(solver=cp.SCS, verbose=False) # SCS est robuste
except:
    prob.solve(verbose=False)

print(f"Status : {prob.status}")

# ==========================================
# 4. AFFICHAGE DÉTAILLÉ
# ==========================================
if prob.status == 'optimal' or prob.status == 'optimal_inaccurate':
    print(f"Coût Minimal (Mouvements) : {prob.value:.2f}")
    print("\n" + "="*60)
    print("DÉTAIL DE LA SOLUTION (PHASE PAR PHASE)")
    print("="*60)

    # Reconstruction numérique des occupations
    # On récupère les valeurs de x pour recalculer proprement
    x_val = x.value
    # On initialise l'occupation courante avec l'état initial
    current_occupancy = init_state.copy()

    for p in range(nb_trans + 1):
        # Affichage de la phase p
        print(f"\n--- PHASE {p} ---")
        if p > 0 and p <= 5:
            closed = renovations[p-1]
            if closed: print(f"(Travaux en cours : {closed})")

        for j in range(nb_bur):
            occupants = []
            for s in range(nb_serv):
                # Valeur numérique
                val = current_occupancy[s, j]
                if val > 0.01: # Seuil d'affichage
                    occupants.append(f"{services[s]}:{val:.2f}")

            if occupants:
                print(f"  Bureau {bureaux[j]:<3} : {', '.join(occupants)}")
            else:
                pass # Bureau vide, on n'affiche rien pour alléger ou : print(f"  Bureau {bureaux[j]:<3} : Vide")

        # Calcul de l'occupation suivante (si pas dernière phase)
        if p < nb_trans:
            next_occupancy = np.zeros((nb_serv, nb_bur))
            for s in range(nb_serv):
                for j in range(nb_bur):
                    # Somme des flux entrants en j à la transition p
                    flux_in = np.sum(x_val[s, p, :, j])
                    next_occupancy[s, j] = flux_in
            current_occupancy = next_occupancy

else:
    print("ERREUR : Le problème est infaisable ou n'a pas convergé.")

Initialisation des données...
Construction du modèle LP (Relaxation Stricte)...
Résolution en cours...
Status : optimal
Coût Minimal (Mouvements) : 30.00

DÉTAIL DE LA SOLUTION (PHASE PAR PHASE)

--- PHASE 0 ---
  Bureau A1  : P:2.00
  Bureau A2  : P:2.00
  Bureau B1  : O:1.00, M:1.00
  Bureau B2  : S:2.00
  Bureau B3  : O:1.00, T:1.00
  Bureau C1  : T:2.00
  Bureau C2  : S:2.00
  Bureau D1  : T:1.00
  Bureau D2  : M:1.00
  Bureau D3  : M:2.00

--- PHASE 1 ---
(Travaux en cours : ['B1', 'B2', 'B3'])
  Bureau A1  : P:1.54, S:0.40, T:0.05
  Bureau A2  : P:1.54, S:0.40, T:0.05
  Bureau C1  : T:2.00
  Bureau C2  : S:1.90, T:0.10
  Bureau D1  : P:0.03, S:0.22, T:1.09, M:0.66
  Bureau D2  : P:0.29, S:0.21, T:0.16, M:1.34
  Bureau D3  : M:2.00
  Bureau E1  : P:0.29, S:0.44, O:1.00, T:0.27
  Bureau E2  : P:0.31, S:0.42, O:1.00, T:0.27

--- PHASE 2 ---
(Travaux en cours : ['D1', 'D2', 'D3'])
  Bureau A1  : P:1.23, S:0.64, T:0.13
  Bureau A2  : P:1.23, S:0.64, T:0.13
  Bureau B1  : P:0.39, S:0.1

In [ ]:
# ============================================================================
# 1. Q3 version MILP
# ============================================================================

bureaux = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3', 'E1', 'E2']
services = ['P', 'S', 'O', 'T', 'M']
nb_bur = 12
nb_serv = 5
nb_trans = 5

# Dictionnaires de mapping
b_idx = {b: i for i, b in enumerate(bureaux)}
s_idx = {s: i for i, s in enumerate(services)}
idx_P, idx_S = s_idx['P'], s_idx['S']

# États Initiaux et Finaux
def build_state(config):
    state = np.zeros((nb_serv, nb_bur))
    for b, servs in config.items():
        for s in servs: state[s_idx[s], b_idx[b]] += 1
    return state

init_config = {
    'A1': ['P', 'P'], 'A2': ['P', 'P'], 'B1': ['O', 'M'], 'B2': ['S', 'S'], 'B3': ['O', 'T'],
    'C1': ['T', 'T'], 'C2': ['S', 'S'], 'D1': ['T'], 'D2': ['M'], 'D3': ['M', 'M']
}
final_config = {
    'A1': ['P', 'P'], 'A2': ['P', 'P'], 'B1': ['M'], 'B2': ['M', 'M'], 'B3': ['M'],
    'C1': ['S', 'S'], 'C2': ['S', 'S'], 'D1': ['T'], 'D2': ['T', 'T'], 'D3': ['T'],
    'E1': ['O'], 'E2': ['O']
}
init_state = build_state(init_config)
final_state = build_state(final_config)

# Contraintes structurelles
renovations = {
    0: ['B1', 'B2', 'B3'], 1: ['D1', 'D2', 'D3'], 2: ['C1', 'C2'],
    3: ['A1', 'A2'], 4: []
}
edges = [
    ('A1', 'D3'), ('D3', 'D2'), ('D2', 'D1'), ('D1', 'C2'),
    ('C2', 'C1'), ('C1', 'B3'), ('B3', 'B2'), ('B2', 'B1'),
    ('B1', 'A2'), ('A2', 'A1'), ('D2', 'E1'), ('E1', 'E2'), ('E2', 'B2')
]

# ============================================================================
# 2. MODÉLISATION (MILP)
# ============================================================================

print("Construction du modèle MILP (Exclusion Stricte)...")

# Variables
# x : Flux de personnes (Continu)
x = cp.Variable((nb_serv, nb_trans, nb_bur, nb_bur), nonneg=True)
# y : Présence binaire (1 si le service occupe le bureau, 0 sinon)
y_P = cp.Variable((nb_trans, nb_bur), boolean=True)
y_S = cp.Variable((nb_trans, nb_bur), boolean=True)

constraints = []
objective_terms = []
M = 2  # Capacité max d'un bureau

# Conservation de flux (Identique au LP)
for s in range(nb_serv):
    for i in range(nb_bur): constraints.append(cp.sum(x[s, 0, i, :]) == init_state[s, i])
    for p in range(nb_trans - 1):
        for k in range(nb_bur): constraints.append(cp.sum(x[s, p, :, k]) == cp.sum(x[s, p+1, k, :]))
    for j in range(nb_bur): constraints.append(cp.sum(x[s, nb_trans-1, :, j]) == final_state[s, j])

# Contraintes par phase
for p in range(nb_trans):
    closed = renovations[p]
    closed_idx = [b_idx[b] for b in closed]

    for j in range(nb_bur):
        # 1. Objectif : Minimiser les mouvements
        flux_mouv = x[:, p, [i for i in range(nb_bur) if i != j], j]
        objective_terms.append(cp.sum(flux_mouv))

        # 2. Capacité & Rénovation
        flux_entrant = cp.sum(x[:, p, :, j]) # Somme de tous les services
        constraints.append(flux_entrant <= 2)

        if j in closed_idx:
            constraints.append(flux_entrant == 0)
            constraints.append(y_P[p, j] == 0)
            constraints.append(y_S[p, j] == 0)

        # 3. Liaison Variable Continue <-> Binaire
        # Si x > 0, alors y doit valoir 1
        occ_P = cp.sum(x[idx_P, p, :, j])
        occ_S = cp.sum(x[idx_S, p, :, j])
        constraints.append(occ_P <= M * y_P[p, j])
        constraints.append(occ_S <= M * y_S[p, j])

        # 4. Exclusion Stricte (Cohabitation)
        # P et S ne peuvent pas être dans le même bureau
        constraints.append(y_P[p, j] + y_S[p, j] <= 1)

    # 5. Exclusion Stricte (Voisinage)
    for (u_name, v_name) in edges:
        u, v = b_idx[u_name], b_idx[v_name]
        if u in closed_idx or v in closed_idx: continue

        # Si P est en u et S en v -> Interdit
        constraints.append(y_P[p, u] + y_S[p, v] <= 1)
        constraints.append(y_S[p, u] + y_P[p, v] <= 1)

# ============================================================================
# 3. RÉSOLUTION
# ============================================================================

print("Résolution en cours...")
prob = cp.Problem(cp.Minimize(cp.sum(objective_terms)), constraints)

# Tentative de résolution avec différents solveurs MIP
solveurs_mip = [cp.GLPK_MI, cp.CBC, cp.SCIP]
solved = False
for solver in solveurs_mip:
    if solver in cp.installed_solvers():
        try:
            prob.solve(solver=solver, verbose=False)
            solved = True
            print(f"   -> Résolu avec {solver}")
            break
        except: pass

if not solved:
    print("   -> Attention : Solveurs MIP dédiés non trouvés. Utilisation du solveur par défaut.")
    try: prob.solve(verbose=False)
    except: pass

# ============================================================================
# 4. AFFICHAGE DES RÉSULTATS
# ============================================================================

if prob.status == 'optimal':
    print(f"\n{'='*60}")
    print(f"RÉSULTAT QUESTION 3 (STRICTE) : {prob.value:.2f} Mouvements")
    print(f"{'='*60}")

    # Reconstitution des phases
    x_val = x.value
    current_state = init_state.copy()

    for p in range(nb_trans + 1):
        print(f"\n--- PHASE {p} ---", end="")
        if p > 0 and p <= 5: print(f" (Travaux: {renovations[p-1]})")
        else: print("")

        for j in range(nb_bur):
            occupants = []
            for s in range(nb_serv):
                val = current_state[s, j]
                if val > 0.01:
                    occupants.append(f"{services[s]}:{val:.2f}")

            if occupants:
                print(f"  Bureau {bureaux[j]:<3} : {', '.join(occupants)}")

        # Mise à jour pour la phase suivante
        if p < nb_trans:
            next_state = np.zeros((nb_serv, nb_bur))
            for s in range(nb_serv):
                for j in range(nb_bur):
                    next_state[s, j] = np.sum(x_val[s, p, :, j])
            current_state = next_state

    # ============================================================================
    # 5. VÉRIFICATION AUTOMATIQUE
    # ============================================================================
    print(f"\n{'='*60}")
    print("VÉRIFICATION DE LA CONFORMITÉ")
    print(f"{'='*60}")

    # On revérifie sur les valeurs numériques finales
    y_P_res = y_P.value
    y_S_res = y_S.value
    sans_conflit = True

    if y_P_res is not None:
        for p in range(nb_trans):
            # Vérif Cohabitation
            for j in range(nb_bur):
                if y_P_res[p, j] > 0.5 and y_S_res[p, j] > 0.5:
                    print(f"ALERTE : P et S cohabitent en {bureaux[j]} à la phase {p}")
                    sans_conflit = False

            # Vérif Voisinage
            for (u_name, v_name) in edges:
                u, v = b_idx[u_name], b_idx[v_name]
                if y_P_res[p, u] > 0.5 and y_S_res[p, v] > 0.5:
                    print(f"ALERTE : P ({u_name}) voisin de S ({v_name}) à la phase {p}")
                    sans_conflit = False
                if y_S_res[p, u] > 0.5 and y_P_res[p, v] > 0.5:
                    print(f"ALERTE : S ({u_name}) voisin de P ({v_name}) à la phase {p}")
                    sans_conflit = False

    if sans_conflit:
        print("SUCCÈS : Aucune violation de la contrainte d'exclusion stricte détectée.")
        print("   La Présidence et les Étudiants ne sont jamais voisins ni colocataires.")
    else:
        print("ÉCHEC : Des conflits existent (voir ci-dessus).")

else:
    print("ERREUR : Le problème n'a pas pu être résolu (Infeasible ou erreur solveur).")

Construction du modèle MILP (Exclusion Stricte)...
Résolution en cours...
   -> Attention : Solveurs MIP dédiés non trouvés. Utilisation du solveur par défaut.

RÉSULTAT QUESTION 3 (STRICTE) : 30.00 Mouvements

--- PHASE 0 ---
  Bureau A1  : P:2.00
  Bureau A2  : P:2.00
  Bureau B1  : O:1.00, M:1.00
  Bureau B2  : S:2.00
  Bureau B3  : O:1.00, T:1.00
  Bureau C1  : T:2.00
  Bureau C2  : S:2.00
  Bureau D1  : T:1.00
  Bureau D2  : M:1.00
  Bureau D3  : M:2.00

--- PHASE 1 --- (Travaux: ['B1', 'B2', 'B3'])
  Bureau A1  : P:2.00
  Bureau A2  : P:2.00
  Bureau C1  : T:2.00
  Bureau C2  : S:2.00
  Bureau D1  : T:1.00, M:1.00
  Bureau D2  : T:1.00, M:1.00
  Bureau D3  : M:2.00
  Bureau E1  : S:1.00, O:1.00
  Bureau E2  : S:1.00, O:1.00

--- PHASE 2 --- (Travaux: ['D1', 'D2', 'D3'])
  Bureau A1  : P:2.00
  Bureau A2  : P:2.00
  Bureau B1  : T:1.00, M:1.00
  Bureau B2  : M:2.00
  Bureau B3  : T:1.00, M:1.00
  Bureau C1  : T:2.00
  Bureau C2  : S:2.00
  Bureau E1  : S:1.00, O:1.00
  Bureau E2  

In [3]:
#Question 4

# ==========================================
# 1. PARAMÈTRES ET DONNÉES
# ==========================================
# (On garde les mêmes données init_state, final_state, etc.)
# bureaux, services, renovations, init_state, final_state, nb_trans...

# Paramètre lambda (poids de la pénalité)
LAMBDA = 100  # Valeur demandée dans la question 5

# ==========================================
# 2. MODÉLISATION AVEC PÉNALITÉ L1
# ==========================================

x = cp.Variable((nb_serv, nb_trans, nb_bur, nb_bur), nonneg=True)

# Variables auxiliaires pour linéariser la valeur absolue |Occup - Final|
# delta[s, p, i] représente l'écart entre l'état actuel et l'état final
delta = cp.Variable((nb_serv, nb_trans, nb_bur), nonneg=True)

constraints = []
mouvements_cost = []
penalty_cost = []

# --- BOUCLE PRINCIPALE ---
for p in range(nb_trans):

    # 1. Coût des Mouvements (Comme avant)
    for j in range(nb_bur):
        for i in range(nb_bur):
            if i != j:
                mouvements_cost.append(cp.sum(x[:, p, i, j]))

        # 2. Contraintes de Capacité & Rénovation (Comme avant)
        constraints.append(cp.sum(x[:, p, :, j]) <= 2)

        if bureaux[j] in renovations[p]:
             constraints.append(cp.sum(x[:, p, :, j]) == 0)

    # 3. CALCUL DE LA PÉNALITÉ
    # On regarde l'état des bureaux APRES la transition p (donc phase p+1)
    # p=0 donne l'état de la Phase 1.
    # L'énoncé demande la somme pour p=0 à 3 (les phases intermédiaires).

    if p < 4: # On ne pénalise pas la dernière transition car elle DOIT être égale au final
        for s in range(nb_serv):
            for j in range(nb_bur):
                # Occupation actuelle du bureau j par le service s à la fin de la transition p
                occup_actuelle = cp.sum(x[s, p, :, j]) # Somme de tout ce qui arrive en j

                # Occupation cible (Finale)
                occup_cible = final_state[s, j]

                # Contraintes pour linéariser |actuel - cible| <= delta
                # delta >= actuel - cible
                constraints.append(delta[s, p, j] >= occup_actuelle - occup_cible)
                # delta >= -(actuel - cible)  =>  delta >= cible - actuel
                constraints.append(delta[s, p, j] >= occup_cible - occup_actuelle)

                # On ajoute ce delta à l'objectif de pénalité
                penalty_cost.append(delta[s, p, j])

# --- CONSERVATION DE FLUX (Identique Q2) ---
for s in range(nb_serv):
    for i in range(nb_bur):
        constraints.append(cp.sum(x[s, 0, i, :]) == init_state[s, i])
    for p in range(nb_trans - 1):
        for k in range(nb_bur):
            constraints.append(cp.sum(x[s, p, :, k]) == cp.sum(x[s, p+1, k, :]))
    for j in range(nb_bur):
        constraints.append(cp.sum(x[s, nb_trans-1, :, j]) == final_state[s, j])

# ==========================================
# 3. RÉSOLUTION ET COMPARAISON MISE À JOUR
# ==========================================

total_objective = cp.sum(mouvements_cost) + LAMBDA * cp.sum(penalty_cost)
prob_q4 = cp.Problem(cp.Minimize(total_objective), constraints)

print(f"Résolution Q4/Q5 avec Lambda = {LAMBDA}...")
try:
    prob_q4.solve()
except:
    prob_q4.solve(solver=cp.SCIPY)

print(f"Status : {prob_q4.status}")

if prob_q4.status == 'optimal':
    val_mouv = cp.sum(mouvements_cost).value
    val_pen = cp.sum(penalty_cost).value
    print(f"Total Objectif : {prob_q4.value:.2f}")
    print(f" -> Dont Coût Mouvements réels : {val_mouv:.2f}")
    print(f" -> Dont Terme de Pénalité (Distance au final) : {val_pen:.2f}")

    print("\nComparaison :")
    # On compare avec 30.00 (le vrai optimum)
    print(f"Sans pénalité (Q2), on avait 30.00 mouvements.")
    print(f"Avec pénalité, on a {val_mouv:.2f} mouvements.")

    if val_mouv > 30.01:
        print("Conclusion : Pour respecter la configuration finale plus tôt, on a accepté de faire plus de déménagements au total.")
    else:
        print("Conclusion : On a réussi à converger plus vite sans augmenter le nombre total de déménagements !")

Résolution Q4/Q5 avec Lambda = 100...
Status : optimal
Total Objectif : 3238.00
 -> Dont Coût Mouvements réels : 38.00
 -> Dont Terme de Pénalité (Distance au final) : 32.00

Comparaison :
Sans pénalité (Q2), on avait 30.00 mouvements.
Avec pénalité, on a 38.00 mouvements.
Conclusion : Pour respecter la configuration finale plus tôt, on a accepté de faire plus de déménagements au total.


In [ ]:
import cvxpy as cp
import numpy as np
import time

# ============================================================================
# SECTION 3 : RELAXATION SDP (Questions 6-8) - VERSION COMPLÈTE
# ============================================================================
print("=" * 70)
print("RELAXATION SDP")
print("=" * 70)

# --- Données du problème ---
bureaux = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3', 'E1', 'E2']
services = ['P', 'S', 'O', 'T', 'M']
nb_bur, nb_serv = 12, 5

# PHASES : 0=initial, 1-4=travaux, 5=final
# Mais le SDP modélise les phases 0-4 (5 phases), la phase 5 est la cible fixe
nb_phases = 5

b_idx = {b: i for i, b in enumerate(bureaux)}
s_idx = {s: i for i, s in enumerate(services)}
idx_P, idx_S = s_idx['P'], s_idx['S']

# Effectifs en PERSONNES
effectifs = {'P': 4, 'S': 4, 'O': 2, 'T': 4, 'M': 4}
eff = np.array([effectifs[s] for s in services])

# ÉTAT INITIAL FIXÉ (Figure 1) - Y[service, bureau] = nb personnes
etat_initial = np.zeros((nb_serv, nb_bur), dtype=int)
# A1: P×2, A2: P×2
etat_initial[s_idx['P'], b_idx['A1']] = 2
etat_initial[s_idx['P'], b_idx['A2']] = 2
# B1: O×1, M×1
etat_initial[s_idx['O'], b_idx['B1']] = 1
etat_initial[s_idx['M'], b_idx['B1']] = 1
# B2: S×2
etat_initial[s_idx['S'], b_idx['B2']] = 2
# B3: O×1, T×1
etat_initial[s_idx['O'], b_idx['B3']] = 1
etat_initial[s_idx['T'], b_idx['B3']] = 1
# C1: T×2
etat_initial[s_idx['T'], b_idx['C1']] = 2
# C2: S×2
etat_initial[s_idx['S'], b_idx['C2']] = 2
# D1: T×1
etat_initial[s_idx['T'], b_idx['D1']] = 1
# D2: M×1
etat_initial[s_idx['M'], b_idx['D2']] = 1
# D3: M×2
etat_initial[s_idx['M'], b_idx['D3']] = 2

# ÉTAT FINAL FIXÉ (Figure 2)
etat_final = np.zeros((nb_serv, nb_bur), dtype=int)
# A1: P×2, A2: P×2
etat_final[s_idx['P'], b_idx['A1']] = 2
etat_final[s_idx['P'], b_idx['A2']] = 2
# B1: M×1, B2: M×2, B3: M×1
etat_final[s_idx['M'], b_idx['B1']] = 1
etat_final[s_idx['M'], b_idx['B2']] = 2
etat_final[s_idx['M'], b_idx['B3']] = 1
# C1: S×2, C2: S×2
etat_final[s_idx['S'], b_idx['C1']] = 2
etat_final[s_idx['S'], b_idx['C2']] = 2
# D1: T×1, D2: T×2, D3: T×1
etat_final[s_idx['T'], b_idx['D1']] = 1
etat_final[s_idx['T'], b_idx['D2']] = 2
etat_final[s_idx['T'], b_idx['D3']] = 1
# E1: O×1, E2: O×1
etat_final[s_idx['O'], b_idx['E1']] = 1
etat_final[s_idx['O'], b_idx['E2']] = 1

# Graphe de voisinage
edges = [
    ('A1', 'D3'), ('D3', 'D2'), ('D2', 'D1'), ('D1', 'C2'),
    ('C2', 'C1'), ('C1', 'B3'), ('B3', 'B2'), ('B2', 'B1'),
    ('B1', 'A2'), ('A2', 'A1'), ('D2', 'E1'), ('E1', 'E2'), ('E2', 'B2')
]
voisins = {b: set() for b in bureaux}
for (bu, bv) in edges:
    voisins[bu].add(bv)
    voisins[bv].add(bu)

# Rénovations par phase
indispos = {
    0: ['E1', 'E2'],         # E pas construit
    1: ['B1', 'B2', 'B3'],   # B en rénovation
    2: ['D1', 'D2', 'D3'],   # D en rénovation
    3: ['C1', 'C2'],         # C en rénovation
    4: ['A1', 'A2']          # A en rénovation
}
indispos_idx = {p: [b_idx[b] for b in l] for p, l in indispos.items()}

def afficher_etat(Y, titre=""):
    """Affiche un état sous forme lisible"""
    if titre:
        print(f"\n{titre}")
    for j in range(nb_bur):
        occupants = []
        for s in range(nb_serv):
            if Y[s, j] > 0:
                occupants.append(f"{services[s]}×{int(Y[s, j])}")
        total = int(np.sum(Y[:, j]))
        occ_str = ", ".join(occupants) if occupants else "vide"
        print(f"  {bureaux[j]}: [{occ_str}] ({total}/2)")

print("\n" + "─"*60)
print("ÉTAT INITIAL (Figure 1) - Phase 0")
print("─"*60)
afficher_etat(etat_initial)
print(f"\nVérification effectifs: {[int(np.sum(etat_initial[s,:])) for s in range(nb_serv)]} = {eff.tolist()}")

print("\n" + "─"*60)
print("ÉTAT FINAL CIBLE (Figure 2) - Après Phase 4")
print("─"*60)
afficher_etat(etat_final)
print(f"\nVérification effectifs: {[int(np.sum(etat_final[s,:])) for s in range(nb_serv)]} = {eff.tolist()}")

# ============================================================================
# QUESTION 6 : FORMULATION SDP
# ============================================================================
print("\n" + "=" * 70)
print("QUESTION 6 : FORMULATION SDP")
print("=" * 70)

# Pour le SDP, on utilise des variables binaires de PRÉSENCE (pas le nombre de personnes)
# y_{s,p,j} = 1 si service s présent dans bureau j à phase p
# Ceci est une RELAXATION du problème réel

def index(s, p, j):
    return 1 + s * (nb_phases * nb_bur) + p * nb_bur + j

N = nb_serv * nb_phases * nb_bur
print(f"\nDimension : N = {N} variables binaires (présence)")
print(f"Matrice U ∈ S^{N+1}")

print("""
Transformation : y ∈ {0,1} -> u ∈ {-1,+1} via u = 2y - 1

Contrainte d'exclusion P-S (y_P × y_S = 0) devient :
  1 + U_{P,0} + U_{S,0} + U_{P,S} = 0

Relaxation : U = uuᵀ (rang 1) -> U ⪰ 0, U_{ii} = 1
""")

U = cp.Variable((N+1, N+1), symmetric=True)
constraints = [U >> 0, cp.diag(U) == 1]

# Calculer le nombre minimum de bureaux par service
besoins_min = np.ceil(eff / 2).astype(int)  # [2, 2, 1, 2, 2]

for p in range(nb_phases):
    # C1. Exclusion P-S (voisinage)
    for (bu, bv) in edges:
        u_i, v_i = b_idx[bu], b_idx[bv]
        idx1, idx2 = index(idx_P, p, u_i), index(idx_S, p, v_i)
        constraints.append(1 + U[idx1, 0] + U[idx2, 0] + U[idx1, idx2] == 0)
        idx3, idx4 = index(idx_S, p, u_i), index(idx_P, p, v_i)
        constraints.append(1 + U[idx3, 0] + U[idx4, 0] + U[idx3, idx4] == 0)

    # C2. Exclusion P-S (cohabitation)
    for j in range(nb_bur):
        idx_Pj, idx_Sj = index(idx_P, p, j), index(idx_S, p, j)
        constraints.append(1 + U[idx_Pj, 0] + U[idx_Sj, 0] + U[idx_Pj, idx_Sj] == 0)

    # C3. Capacité (au plus 2 services par bureau)
    for j in range(nb_bur):
        sum_u = sum(U[index(s, p, j), 0] for s in range(nb_serv))
        constraints.append(sum_u <= 2*2 - nb_serv)

    # C4. Effectifs minimum (présence dans au moins k bureaux)
    for s in range(nb_serv):
        sum_u = sum(U[index(s, p, j), 0] for j in range(nb_bur))
        constraints.append(sum_u >= 2*besoins_min[s] - nb_bur)

    # C5. Rénovations
    for j in indispos_idx.get(p, []):
        for s in range(nb_serv):
            constraints.append(U[index(s, p, j), 0] == -1)

# C6. FIXER L'ÉTAT INITIAL (Phase 0)
for s in range(nb_serv):
    for j in range(nb_bur):
        if etat_initial[s, j] > 0:
            constraints.append(U[index(s, 0, j), 0] == 1)  # présent
        else:
            constraints.append(U[index(s, 0, j), 0] == -1)  # absent

# Objectif : minimiser mouvements (stabilité inter-phases) + distance à la cible
obj_terms = []

# Stabilité inter-phases
for s in range(nb_serv):
    for p in range(nb_phases - 1):
        for j in range(nb_bur):
            obj_terms.append(-U[index(s, p, j), index(s, p+1, j)])

# Distance à l'état final (présence)
for s in range(nb_serv):
    for j in range(nb_bur):
        cible = 1.0 if etat_final[s, j] > 0 else -1.0
        obj_terms.append(-cible * U[index(s, nb_phases-1, j), 0])

prob = cp.Problem(cp.Minimize(cp.sum(obj_terms)), constraints)

# ============================================================================
# QUESTION 7 : RÉSOLUTION NUMÉRIQUE
# ============================================================================
print("\n" + "=" * 70)
print("QUESTION 7 : RÉSOLUTION DU SDP")
print("=" * 70)

start = time.time()
prob.solve(solver=cp.SCS, verbose=True, max_iters=5000)
temps = time.time() - start

print(f"\n{'─'*60}")
print(f"Status : {prob.status}")
print(f"Valeur optimale : {prob.value:.2f}")
print(f"Temps : {temps:.1f}s")

U_val = U.value
vals, vecs = np.linalg.eigh(U_val)
rang = np.sum(vals > 1e-4)
print(f"Rang effectif de U* : {rang} / {N+1}")

# Extraction des probabilités de présence (solution fractionnaire)
y_frac = np.zeros((nb_serv, nb_phases, nb_bur))
for s in range(nb_serv):
    for p in range(nb_phases):
        for j in range(nb_bur):
            y_frac[s, p, j] = (1 + U_val[index(s, p, j), 0]) / 2

print(f"\n{'─'*60}")
print("SOLUTION FRACTIONNAIRE (probabilités de présence)")
print(f"{'─'*60}")

for p in range(nb_phases):
    print(f"\nPhase {p} (fermés: {indispos.get(p, [])}):")
    for s in range(nb_serv):
        probs = [(bureaux[j], y_frac[s,p,j]) for j in range(nb_bur) if y_frac[s,p,j] > 0.01]
        probs.sort(key=lambda x: -x[1])
        probs_str = ", ".join([f"{b}:{v:.2f}" for b,v in probs[:5]])
        print(f"  {services[s]}: {probs_str}")

# Mouvements fractionnaires
print(f"\n{'─'*60}")
print("MOUVEMENTS FRACTIONNAIRES (borne inférieure)")
print(f"{'─'*60}")

y_init_bin = (etat_initial > 0).astype(float)
y_final_bin = (etat_final > 0).astype(float)

total_frac = 0
# Phase 0 est fixée, donc on compte à partir de 0->1
for p in range(nb_phases - 1):
    mvt = sum(np.sum(np.abs(y_frac[s, p, :] - y_frac[s, p+1, :])) / 2 for s in range(nb_serv))
    print(f"  Phase {p}->{p+1} : {mvt:.2f}")
    total_frac += mvt

mvt_final = sum(np.sum(np.abs(y_frac[s, nb_phases-1, :] - y_final_bin[s, :])) / 2 for s in range(nb_serv))
print(f"  Phase 4->Final : {mvt_final:.2f}")
total_frac += mvt_final

print(f"{'─'*60}")
print(f"  TOTAL FRACTIONNAIRE : {total_frac:.2f}")
print("TOTAL FRACTIONNAIRE", total_frac*2.0)

# ============================================================================
# QUESTION 8 : ARRONDI RANDOMISÉ + RÉPARATION
# ============================================================================
print("\n" + "=" * 70)
print("QUESTION 8 : ARRONDI RANDOMISÉ + RÉPARATION")
print("=" * 70)

print("""
Méthode :
1. Décomposer U* = VVᵀ
2. Projeter sur un vecteur aléatoire r
3. Utiliser les scores pour guider une réparation
4. La réparation utilise Y ∈ {0,1,2} (personnes) pour satisfaire les effectifs exacts

Note : La Phase 0 est FIXÉE à l'état initial.
""")

vals[vals < 0] = 0
V = vecs @ np.diag(np.sqrt(vals))

def reparer_phase_personnes(scores, p, Y_prec=None):
    """
    Répare une phase avec le modèle PERSONNES (Y ∈ {0,1,2}).
    Si p=0, retourne l'état initial.
    """
    if p == 0:
        return etat_initial.copy()

    Y = np.zeros((nb_serv, nb_bur), dtype=int)
    dispo = [j for j in range(nb_bur) if j not in indispos_idx.get(p, [])]
    capacite = {j: 2 for j in dispo}

    # Ordre : P et S d'abord
    for s in [idx_P, idx_S]:
        restant = eff[s]
        candidats = sorted([(scores[s, j], j) for j in dispo], reverse=True)

        for (_, j) in candidats:
            if restant <= 0:
                break
            if capacite[j] <= 0:
                continue

            # Exclusion P-S
            if s == idx_P and Y[idx_S, j] > 0:
                continue
            if s == idx_S and Y[idx_P, j] > 0:
                continue

            conflit = False
            for v in voisins[bureaux[j]]:
                vi = b_idx[v]
                if vi in dispo:
                    if s == idx_P and Y[idx_S, vi] > 0:
                        conflit = True; break
                    if s == idx_S and Y[idx_P, vi] > 0:
                        conflit = True; break
            if conflit:
                continue

            assign = min(restant, capacite[j], 2)
            Y[s, j] = assign
            capacite[j] -= assign
            restant -= assign

        if restant > 0:
            return None

    # Puis O, T, M
    for s in range(nb_serv):
        if s in [idx_P, idx_S]:
            continue
        restant = eff[s]
        candidats = sorted([(scores[s, j], j) for j in dispo], reverse=True)

        for (_, j) in candidats:
            if restant <= 0:
                break
            if capacite[j] <= 0:
                continue
            assign = min(restant, capacite[j], 2 - Y[s, j])
            Y[s, j] += assign
            capacite[j] -= assign
            restant -= assign

        if restant > 0:
            return None

    return Y

def verifier_solution(Y_all):
    """Vérifie toutes les contraintes"""
    for p in range(nb_phases):
        Y = Y_all[:, p, :]

        # Bureaux fermés
        for j in indispos_idx.get(p, []):
            if np.sum(Y[:, j]) > 0:
                return False, f"Phase {p}: bureau fermé {bureaux[j]} occupé"

        # Effectifs
        for s in range(nb_serv):
            if np.sum(Y[s, :]) != eff[s]:
                return False, f"Phase {p}: effectif {services[s]} = {np.sum(Y[s,:])} ≠ {eff[s]}"

        # Capacité
        for j in range(nb_bur):
            if np.sum(Y[:, j]) > 2:
                return False, f"Phase {p}: capacité {bureaux[j]} dépassée"

        # Exclusion P-S
        for j in range(nb_bur):
            if Y[idx_P, j] > 0 and Y[idx_S, j] > 0:
                return False, f"Phase {p}: cohabitation P-S en {bureaux[j]}"
            if Y[idx_P, j] > 0:
                for v in voisins[bureaux[j]]:
                    if Y[idx_S, b_idx[v]] > 0:
                        return False, f"Phase {p}: voisinage P-S"

    # Vérifier que Phase 0 = état initial
    if not np.array_equal(Y_all[:, 0, :], etat_initial):
        return False, "Phase 0 ≠ état initial"

    return True, "OK"

def calculer_mouvements_personnes(Y_all):
    """Calcule les mouvements en PERSONNES"""
    total = 0
    details = []

    for p in range(nb_phases - 1):
        mvt = 0
        for s in range(nb_serv):
            mvt += np.sum(np.maximum(Y_all[s, p, :] - Y_all[s, p+1, :], 0))
        details.append(int(mvt))
        total += mvt

    # Vers état final
    mvt_final = 0
    for s in range(nb_serv):
        mvt_final += np.sum(np.maximum(Y_all[s, nb_phases-1, :] - etat_final[s, :], 0))
    details.append(int(mvt_final))
    total += mvt_final

    return int(total), details

# Arrondi randomisé
print("\nRecherche de solutions entières...")
nb_trials = 10000
solutions = []
np.random.seed(42)
t0 = time.time()

for trial in range(nb_trials):
    if trial % 2000 == 0:
        print(f"  Essai {trial}... ({len(solutions)} solutions)")

    r = np.random.randn(V.shape[1])
    proj = V @ r
    if proj[0] < 0:
        proj = -proj

    scores = np.zeros((nb_serv, nb_phases, nb_bur))
    for s in range(nb_serv):
        for p in range(nb_phases):
            for j in range(nb_bur):
                scores[s, p, j] = proj[index(s, p, j)]

    scores += np.random.randn(nb_serv, nb_phases, nb_bur) * 0.3

    Y_all = np.zeros((nb_serv, nb_phases, nb_bur), dtype=int)
    ok = True

    for p in range(nb_phases):
        Y_p = reparer_phase_personnes(scores[:, p, :], p)
        if Y_p is None:
            ok = False
            break
        Y_all[:, p, :] = Y_p

    if ok:
        valid, msg = verifier_solution(Y_all)
        if valid:
            mvt, details = calculer_mouvements_personnes(Y_all)
            solutions.append({'Y': Y_all.copy(), 'mvt': mvt, 'details': details})

print(f"\nTemps : {time.time()-t0:.1f}s")

# ============================================================================
# RÉSULTATS
# ============================================================================
print(f"\n{'─'*60}")
print(f"Solutions trouvées : {len(solutions)} / {nb_trials}")

if solutions:
    mouvements = [sol['mvt'] for sol in solutions]
    print(f"  Min: {min(mouvements)}, Max: {max(mouvements)}, Moy: {np.mean(mouvements):.1f}")

    best = min(solutions, key=lambda x: x['mvt'])
    Y_best = best['Y']

    print(f"\n{'='*60}")
    print(f"MEILLEURE SOLUTION ENTIÈRE : {best['mvt']} MOUVEMENTS")
    print(f"{'='*60}")

    # Afficher toutes les phases
    for p in range(nb_phases):
        print(f"\n{'─'*60}")
        print(f"PHASE {p} (fermés: {indispos.get(p, [])})")
        print(f"{'─'*60}")
        afficher_etat(Y_best[:, p, :])

        # Vérification P-S
        P_burs = [bureaux[j] for j in range(nb_bur) if Y_best[idx_P, p, j] > 0]
        S_burs = [bureaux[j] for j in range(nb_bur) if Y_best[idx_S, p, j] > 0]
        print(f"\n  P dans: {P_burs}")
        print(f"  S dans: {S_burs}")
        print(f"  Exclusion P-S: ✓")

    # Afficher l'état final cible
    print(f"\n{'─'*60}")
    print(f"ÉTAT FINAL CIBLE (Figure 2)")
    print(f"{'─'*60}")
    afficher_etat(etat_final)

    # Détail des mouvements
    print(f"\n{'─'*60}")
    print("DÉTAIL DES MOUVEMENTS (en personnes)")
    print(f"{'─'*60}")

    for p in range(nb_phases - 1):
        print(f"\n  Phase {p} -> Phase {p+1} : {best['details'][p]} personnes")
        for s in range(nb_serv):
            Y_before = Y_best[s, p, :]
            Y_after = Y_best[s, p+1, :]
            departs = []
            arrivees = []
            for j in range(nb_bur):
                diff = Y_after[j] - Y_before[j]
                if diff < 0:
                    departs.append(f"{bureaux[j]}({-diff})")
                elif diff > 0:
                    arrivees.append(f"{bureaux[j]}(+{diff})")
            if departs or arrivees:
                print(f"    {services[s]}: quitte {departs} -> arrive {arrivees}")

    print(f"\n  Phase 4 -> État Final : {best['details'][-1]} personnes")
    for s in range(nb_serv):
        Y_last = Y_best[s, nb_phases-1, :]
        Y_cible = etat_final[s, :]
        departs = []
        arrivees = []
        for j in range(nb_bur):
            diff = Y_cible[j] - Y_last[j]
            if diff < 0:
                departs.append(f"{bureaux[j]}({-diff})")
            elif diff > 0:
                arrivees.append(f"{bureaux[j]}(+{diff})")
        if departs or arrivees:
            print(f"    {services[s]}: quitte {departs} -> arrive {arrivees}")

    print(f"\n{'='*60}")
    print(f"TOTAL : {best['mvt']} MOUVEMENTS DE PERSONNES")
    print(f"{'='*60}")

    # Comparaison
    print(f"\n{'─'*60}")
    print("COMPARAISON ET ANALYSE")
    print(f"{'─'*60}")
    print(f"  Borne SDP fractionnaire : {total_frac:.1f} mouvements")
    print(f"  Solution entière        : {best['mvt']} mouvements")
    print(f"  Gap d'intégrité         : {best['mvt'] - total_frac:.0f} ({100*(best['mvt']-total_frac)/max(total_frac,0.1):.0f}%)")

    print(f"""
  Analyse :
  - La Phase 0 est FIXÉE à l'état initial (Figure 1)
  - L'état final cible est celui de la Figure 2
  - Le SDP modélise des PRÉSENCES binaires (relaxation)
  - La réparation assigne des PERSONNES (Y ∈ {{0,1,2}})

  Conclusion Question 8 :
  - L'arrondi randomisé direct ne donne pas de solutions valides
  - Avec réparation heuristique : {len(solutions)}/{nb_trials} solutions
  - Solution RÉALISABLE mais NON GARANTIE OPTIMALE
  - L'optimal est dans l'intervalle [{total_frac:.0f}, {best['mvt']}]
""")

else:
    print("\n Aucune solution trouvée")

(CVXPY) Jan 28 02:49:07 PM: Your problem has 90601 variables, 91297 constraints, and 0 parameters.
(CVXPY) Jan 28 02:49:07 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jan 28 02:49:07 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jan 28 02:49:07 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jan 28 02:49:07 PM: Your problem is compiled with the CPP canonicalization backend.


RELAXATION SDP - PROBLÈME PARIS-DUCHESSE

────────────────────────────────────────────────────────────
ÉTAT INITIAL (Figure 1) - Phase 0
────────────────────────────────────────────────────────────
  A1: [P×2] (2/2)
  A2: [P×2] (2/2)
  B1: [O×1, M×1] (2/2)
  B2: [S×2] (2/2)
  B3: [O×1, T×1] (2/2)
  C1: [T×2] (2/2)
  C2: [S×2] (2/2)
  D1: [T×1] (1/2)
  D2: [M×1] (1/2)
  D3: [M×2] (2/2)
  E1: [vide] (0/2)
  E2: [vide] (0/2)

Vérification effectifs: [4, 4, 2, 4, 4] = [4, 4, 2, 4, 4]

────────────────────────────────────────────────────────────
ÉTAT FINAL CIBLE (Figure 2) - Après Phase 4
────────────────────────────────────────────────────────────
  A1: [P×2] (2/2)
  A2: [P×2] (2/2)
  B1: [M×1] (1/2)
  B2: [M×2] (2/2)
  B3: [M×1] (1/2)
  C1: [S×2] (2/2)
  C2: [S×2] (2/2)
  D1: [T×1] (1/2)
  D2: [T×2] (2/2)
  D3: [T×1] (1/2)
  E1: [O×1] (1/2)
  E2: [O×1] (1/2)

Vérification effectifs: [4, 4, 2, 4, 4] = [4, 4, 2, 4, 4]

QUESTION 6 : FORMULATION SDP

Dimension : N = 300 variables binaires (pr

(CVXPY) Jan 28 02:49:07 PM: Compiling problem (target solver=SCS).
(CVXPY) Jan 28 02:49:07 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Jan 28 02:49:07 PM: Applying reduction Dcp2Cone
(CVXPY) Jan 28 02:49:07 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jan 28 02:49:07 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Jan 28 02:49:28 PM: Applying reduction SCS
(CVXPY) Jan 28 02:49:29 PM: Finished problem compilation (took 2.152e+01 seconds).
(CVXPY) Jan 28 02:49:29 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.11 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 45451, constraints m: 46147
cones: 	  z: primal zero / dual free vars: 611
	  l: linear vars: 85
	  s: psd vars: 45451, ssize: 1
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 5000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 47042, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res 

C:\Users\Magomed Tsitsiev\anaconda3\Lib\site-packages\cvxpy\problems\problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
(CVXPY) Jan 28 02:54:29 PM: Problem status: optimal_inaccurate
(CVXPY) Jan 28 02:54:29 PM: Optimal value: -2.468e+02
(CVXPY) Jan 28 02:54:29 PM: Compilation took 2.152e+01 seconds
(CVXPY) Jan 28 02:54:29 PM: Solver (including time spent in interface) took 3.000e+02 seconds


  5000| 5.31e-03  1.10e-04  8.68e-03 -2.47e+02  1.08e+00  3.00e+02 
------------------------------------------------------------------
status:  solved (inaccurate - reached max_iters)
timings: total: 3.00e+02s = setup: 7.26e-02s + solve: 3.00e+02s
	 lin-sys: 8.89e+00s, cones: 2.88e+02s, accel: 7.29e-01s
------------------------------------------------------------------
objective = -246.768811 (inaccurate)
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------

────────────────────────────────────────────────────────────
Status : optimal_inaccurate
Valeur optimale : -246.77
Temps : 321.6s
Rang effectif de U* : 11 / 301

────────────────────────────────────────────────────────────
SOLUTION FRACTIONNAIRE (probabilités de présence)
──────────

In [4]:

# ============================================================================
# SECTION 3 : RELAXATION SDP (Questions 6-8) - VERSION COMPLÈTE
# ============================================================================

import cvxpy as cp
import numpy as np
import time

# ============================================================================
# DONNÉES DU PROBLÈME
# ============================================================================

print("=" * 70)
print("RELAXATION SDP - PROBLÈME PARIS-DUCHESSE")
print("=" * 70)

# Liste des bureaux et services
bureaux = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3', 'E1', 'E2']
services = ['P', 'S', 'O', 'T', 'M']  # P=Présidence, S=Étudiants, O=Optim, T=Théo, M=Maths
nb_bur, nb_serv = 12, 5

# Nombre de phases modélisées dans le SDP (0 à 4)
# La phase 5 (état final) est une cible fixe, non une variable
nb_phases = 5

# Dictionnaires d'indexation pour accès rapide
b_idx = {b: i for i, b in enumerate(bureaux)}
s_idx = {s: i for i, s in enumerate(services)}
idx_P, idx_S = s_idx['P'], s_idx['S']

# Effectifs de chaque service (en nombre de personnes)
effectifs = {'P': 4, 'S': 4, 'O': 2, 'T': 4, 'M': 4}
eff = np.array([effectifs[s] for s in services])

# ----------------------------------------------------------------------------
# ÉTAT INITIAL (Figure 1 du sujet)
# Y[service, bureau] = nombre de personnes du service dans ce bureau
# ----------------------------------------------------------------------------
etat_initial = np.zeros((nb_serv, nb_bur), dtype=int)

# Aile A : Présidence
etat_initial[s_idx['P'], b_idx['A1']] = 2
etat_initial[s_idx['P'], b_idx['A2']] = 2

# Aile B : Bureaux partagés
etat_initial[s_idx['O'], b_idx['B1']] = 1
etat_initial[s_idx['M'], b_idx['B1']] = 1
etat_initial[s_idx['S'], b_idx['B2']] = 2
etat_initial[s_idx['O'], b_idx['B3']] = 1
etat_initial[s_idx['T'], b_idx['B3']] = 1

# Aile C : Étudiants et Théorique
etat_initial[s_idx['T'], b_idx['C1']] = 2
etat_initial[s_idx['S'], b_idx['C2']] = 2

# Aile D : Théorique et Maths
etat_initial[s_idx['T'], b_idx['D1']] = 1
etat_initial[s_idx['M'], b_idx['D2']] = 1
etat_initial[s_idx['M'], b_idx['D3']] = 2

# ----------------------------------------------------------------------------
# ÉTAT FINAL CIBLE (Figure 2 du sujet)
# ----------------------------------------------------------------------------
etat_final = np.zeros((nb_serv, nb_bur), dtype=int)

# Aile A : Présidence (retour après rénovation)
etat_final[s_idx['P'], b_idx['A1']] = 2
etat_final[s_idx['P'], b_idx['A2']] = 2

# Aile B : Mathématiques
etat_final[s_idx['M'], b_idx['B1']] = 1
etat_final[s_idx['M'], b_idx['B2']] = 2
etat_final[s_idx['M'], b_idx['B3']] = 1

# Aile C : Étudiants
etat_final[s_idx['S'], b_idx['C1']] = 2
etat_final[s_idx['S'], b_idx['C2']] = 2

# Aile D : Théorique
etat_final[s_idx['T'], b_idx['D1']] = 1
etat_final[s_idx['T'], b_idx['D2']] = 2
etat_final[s_idx['T'], b_idx['D3']] = 1

# Aile E : Optimisation
etat_final[s_idx['O'], b_idx['E1']] = 1
etat_final[s_idx['O'], b_idx['E2']] = 1

# ----------------------------------------------------------------------------
# GRAPHE DE VOISINAGE DES BUREAUX
# Utilisé pour la contrainte d'exclusion P-S
# ----------------------------------------------------------------------------
edges = [
    ('A1', 'D3'), ('D3', 'D2'), ('D2', 'D1'), ('D1', 'C2'),
    ('C2', 'C1'), ('C1', 'B3'), ('B3', 'B2'), ('B2', 'B1'),
    ('B1', 'A2'), ('A2', 'A1'), ('D2', 'E1'), ('E1', 'E2'), ('E2', 'B2')
]

voisins = {b: set() for b in bureaux}
for (bu, bv) in edges:
    voisins[bu].add(bv)
    voisins[bv].add(bu)

# ----------------------------------------------------------------------------
# CALENDRIER DES RÉNOVATIONS
# Phase p : liste des bureaux indisponibles
# ----------------------------------------------------------------------------
indispos = {
    0: ['E1', 'E2'],         # Phase 0 : Aile E pas encore construite
    1: ['B1', 'B2', 'B3'],   # Phase 1 : Aile B en rénovation
    2: ['D1', 'D2', 'D3'],   # Phase 2 : Aile D en rénovation
    3: ['C1', 'C2'],         # Phase 3 : Aile C en rénovation
    4: ['A1', 'A2']          # Phase 4 : Aile A en rénovation
}
indispos_idx = {p: [b_idx[b] for b in l] for p, l in indispos.items()}


# ============================================================================
# FONCTIONS UTILITAIRES
# ============================================================================

def afficher_etat(Y, titre=""):
    """
    Affiche l'état d'occupation des bureaux de manière lisible.

    Paramètres:
        Y : np.array de shape (nb_serv, nb_bur) - matrice d'occupation
        titre : str - titre optionnel à afficher
    """
    if titre:
        print(f"\n{titre}")
    for j in range(nb_bur):
        occupants = []
        for s in range(nb_serv):
            if Y[s, j] > 0:
                occupants.append(f"{services[s]}×{int(Y[s, j])}")
        total = int(np.sum(Y[:, j]))
        occ_str = ", ".join(occupants) if occupants else "vide"
        print(f"  {bureaux[j]}: [{occ_str}] ({total}/2)")


# Affichage des états de référence
print("\n" + "─" * 60)
print("ÉTAT INITIAL (Figure 1) - Phase 0")
print("─" * 60)
afficher_etat(etat_initial)
print(f"\nVérification effectifs: {[int(np.sum(etat_initial[s, :])) for s in range(nb_serv)]} = {eff.tolist()}")

print("\n" + "─" * 60)
print("ÉTAT FINAL CIBLE (Figure 2) - Après Phase 4")
print("─" * 60)
afficher_etat(etat_final)
print(f"\nVérification effectifs: {[int(np.sum(etat_final[s, :])) for s in range(nb_serv)]} = {eff.tolist()}")


# ============================================================================
# QUESTION 6 : FORMULATION DU PROGRAMME SDP
# ============================================================================

print("\n" + "=" * 70)
print("QUESTION 6 : FORMULATION SDP")
print("=" * 70)

# ----------------------------------------------------------------------------
# Choix de modélisation : variables binaires de PRÉSENCE
#
# y_{s,p,j} ∈ {0,1} indique si le service s est PRÉSENT dans le bureau j
# à la phase p, sans distinguer si 1 ou 2 personnes l'occupent.
#
# Ce choix réduit la dimension du problème (300 variables au lieu de 1080)
# tout en préservant la structure combinatoire essentielle.
# ----------------------------------------------------------------------------

def index(s, p, j):
    """
    Calcule l'indice linéaire de la variable y_{s,p,j} dans la matrice U.
    L'indice 0 est réservé pour la variable auxiliaire (u_0 = 1).
    """
    return 1 + s * (nb_phases * nb_bur) + p * nb_bur + j


N = nb_serv * nb_phases * nb_bur
print(f"\nDimension : N = {N} variables binaires (présence)")
print(f"Matrice U ∈ S^{N + 1}")

print("""
Transformation : y ∈ {0,1} -> u ∈ {-1,+1} via u = 2y - 1

Contrainte d'exclusion P-S (y_P × y_S = 0) devient :
  1 + U_{P,0} + U_{S,0} + U_{P,S} = 0

Relaxation : U = uuᵀ (rang 1) -> U ⪰ 0, U_{ii} = 1
""")

# ----------------------------------------------------------------------------
# Construction du problème SDP
# ----------------------------------------------------------------------------

# Variable matricielle symétrique semi-définie positive
U = cp.Variable((N + 1, N + 1), symmetric=True)

# Contraintes de base de la relaxation SDP
constraints = [
    U >> 0,                    # U semi-définie positive
    cp.diag(U) == 1            # Diagonale = 1 (u_i² = 1)
]

# Nombre minimum de bureaux par service (plafond de effectif/2)
besoins_min = np.ceil(eff / 2).astype(int)  # [2, 2, 1, 2, 2]

for p in range(nb_phases):

    # C1. EXCLUSION P-S : VOISINAGE
    # P et S ne peuvent pas être dans des bureaux adjacents
    for (bu, bv) in edges:
        u_i, v_i = b_idx[bu], b_idx[bv]
        # Si P dans bu, alors S pas dans bv
        idx1, idx2 = index(idx_P, p, u_i), index(idx_S, p, v_i)
        constraints.append(1 + U[idx1, 0] + U[idx2, 0] + U[idx1, idx2] == 0)
        # Si S dans bu, alors P pas dans bv
        idx3, idx4 = index(idx_S, p, u_i), index(idx_P, p, v_i)
        constraints.append(1 + U[idx3, 0] + U[idx4, 0] + U[idx3, idx4] == 0)

    # C2. EXCLUSION P-S : COHABITATION (contrainte renforcée)
    # P et S ne peuvent pas partager le même bureau
    for j in range(nb_bur):
        idx_Pj, idx_Sj = index(idx_P, p, j), index(idx_S, p, j)
        constraints.append(1 + U[idx_Pj, 0] + U[idx_Sj, 0] + U[idx_Pj, idx_Sj] == 0)

    # C3. CAPACITÉ : au plus 2 services par bureau
    # Σ y_{s,p,j} ≤ 2  ->  Σ (1+u)/2 ≤ 2  ->  Σ u ≤ 4 - nb_serv
    for j in range(nb_bur):
        sum_u = sum(U[index(s, p, j), 0] for s in range(nb_serv))
        constraints.append(sum_u <= 2 * 2 - nb_serv)

    # C4. EFFECTIFS : chaque service occupe au moins k bureaux
    # Σ_j y_{s,p,j} ≥ besoins_min[s]
    for s in range(nb_serv):
        sum_u = sum(U[index(s, p, j), 0] for j in range(nb_bur))
        constraints.append(sum_u >= 2 * besoins_min[s] - nb_bur)

    # C5. RÉNOVATIONS : bureaux fermés inoccupés
    # y_{s,p,j} = 0 pour j fermé  ->  u = -1
    for j in indispos_idx.get(p, []):
        for s in range(nb_serv):
            constraints.append(U[index(s, p, j), 0] == -1)

# C6. ÉTAT INITIAL FIXÉ (Phase 0 = Figure 1)
# Les variables de la phase 0 sont fixées selon l'état initial
for s in range(nb_serv):
    for j in range(nb_bur):
        if etat_initial[s, j] > 0:
            constraints.append(U[index(s, 0, j), 0] == 1)   # Service présent
        else:
            constraints.append(U[index(s, 0, j), 0] == -1)  # Service absent

# ----------------------------------------------------------------------------
# FONCTION OBJECTIF
# Deux composantes : stabilité inter-phases + proximité à l'état final
# ----------------------------------------------------------------------------

obj_terms = []

# Terme 1 : STABILITÉ INTER-PHASES
# Minimiser les changements entre phases consécutives
# On maximise Σ U[y_{s,p,j}, y_{s,p+1,j}] (= 1 si même valeur, -1 sinon)
for s in range(nb_serv):
    for p in range(nb_phases - 1):
        for j in range(nb_bur):
            obj_terms.append(-U[index(s, p, j), index(s, p + 1, j)])

# Terme 2 : PROXIMITÉ À L'ÉTAT FINAL
# Encourager la phase 4 à ressembler à l'état final cible
for s in range(nb_serv):
    for j in range(nb_bur):
        cible = 1.0 if etat_final[s, j] > 0 else -1.0
        obj_terms.append(-cible * U[index(s, nb_phases - 1, j), 0])

# Construction du problème d'optimisation
prob = cp.Problem(cp.Minimize(cp.sum(obj_terms)), constraints)


# ============================================================================
# QUESTION 7 : RÉSOLUTION NUMÉRIQUE DU SDP
# ============================================================================

print("\n" + "=" * 70)
print("QUESTION 7 : RÉSOLUTION DU SDP")
print("=" * 70)

# Résolution avec le solveur SCS
# Note : max_iters=5000 donne un bon compromis entre qualité de la borne
# et facilité d'arrondi (voir discussion dans le rapport)
start = time.time()
prob.solve(solver=cp.SCS, verbose=True, max_iters=5000)
temps = time.time() - start

print(f"\n{'─' * 60}")
print(f"Status : {prob.status}")
print(f"Valeur optimale : {prob.value:.2f}")
print(f"Temps : {temps:.1f}s")

# Analyse spectrale de la solution
U_val = U.value
vals, vecs = np.linalg.eigh(U_val)
rang = np.sum(vals > 1e-4)
print(f"Rang effectif de U* : {rang} / {N + 1}")

# ----------------------------------------------------------------------------
# Extraction de la solution fractionnaire
# Probabilités de présence : ȳ_{s,p,j} = (1 + U_{(s,p,j),0}) / 2
# ----------------------------------------------------------------------------

y_frac = np.zeros((nb_serv, nb_phases, nb_bur))
for s in range(nb_serv):
    for p in range(nb_phases):
        for j in range(nb_bur):
            y_frac[s, p, j] = (1 + U_val[index(s, p, j), 0]) / 2

print(f"\n{'─' * 60}")
print("SOLUTION FRACTIONNAIRE (probabilités de présence)")
print(f"{'─' * 60}")

for p in range(nb_phases):
    print(f"\nPhase {p} (fermés: {indispos.get(p, [])}):")
    for s in range(nb_serv):
        # Afficher les bureaux où le service a une probabilité > 1%
        probs = [(bureaux[j], y_frac[s, p, j]) for j in range(nb_bur) if y_frac[s, p, j] > 0.01]
        probs.sort(key=lambda x: -x[1])
        probs_str = ", ".join([f"{b}:{v:.2f}" for b, v in probs[:5]])
        print(f"  {services[s]}: {probs_str}")

# ----------------------------------------------------------------------------
# Calcul des mouvements fractionnaires (borne inférieure)
# ----------------------------------------------------------------------------

print(f"\n{'─' * 60}")
print("MOUVEMENTS FRACTIONNAIRES (borne inférieure)")
print(f"{'─' * 60}")

# Conversion des états en présence binaire pour comparaison
y_init_bin = (etat_initial > 0).astype(float)
y_final_bin = (etat_final > 0).astype(float)

total_frac = 0

# Mouvements entre phases consécutives
for p in range(nb_phases - 1):
    mvt = sum(np.sum(np.abs(y_frac[s, p, :] - y_frac[s, p + 1, :])) / 2 for s in range(nb_serv))
    print(f"  Phase {p}->{p + 1} : {mvt:.2f}")
    total_frac += mvt

# Mouvements vers l'état final
mvt_final = sum(np.sum(np.abs(y_frac[s, nb_phases - 1, :] - y_final_bin[s, :])) / 2 for s in range(nb_serv))
print(f"  Phase 4->Final : {mvt_final:.2f}")
total_frac += mvt_final

print(f"{'─' * 60}")
print(f"  TOTAL FRACTIONNAIRE : {total_frac:.2f}")
print(f"  ESTIMATION EN PERSONNES : {total_frac * 2:.2f} (×2 car présence -> personnes)")


# ============================================================================
# QUESTION 8 : ARRONDI RANDOMISÉ AVEC RÉPARATION
# ============================================================================

print("\n" + "=" * 70)
print("QUESTION 8 : ARRONDI RANDOMISÉ + RÉPARATION")
print("=" * 70)

print("""
Méthode :
1. Décomposer U* = VVᵀ (Cholesky sur les valeurs propres)
2. Projeter sur un vecteur aléatoire r ~ N(0, I)
3. Utiliser les scores (V·r) pour guider une heuristique de réparation
4. La réparation construit Y ∈ {0,1,2} pour satisfaire les effectifs exacts

Note : La Phase 0 est FIXÉE à l'état initial (Figure 1).
""")

# Décomposition de Cholesky de U*
vals[vals < 0] = 0  # Correction numérique des valeurs propres négatives
V = vecs @ np.diag(np.sqrt(vals))


def reparer_phase_personnes(scores, p, Y_prec=None):
    """
    Construit une allocation valide pour la phase p en utilisant les scores SDP.

    La réparation utilise un algorithme glouton qui place les services
    par ordre de priorité (P et S d'abord pour l'exclusion) dans les bureaux
    ayant les meilleurs scores, tout en respectant toutes les contraintes.

    Paramètres:
        scores : np.array (nb_serv, nb_bur) - scores de préférence issus du SDP
        p : int - numéro de la phase
        Y_prec : np.array - allocation de la phase précédente (non utilisé ici)

    Retourne:
        Y : np.array (nb_serv, nb_bur) - allocation valide, ou None si impossible
    """
    # Phase 0 : retourner l'état initial fixé
    if p == 0:
        return etat_initial.copy()

    Y = np.zeros((nb_serv, nb_bur), dtype=int)

    # Bureaux disponibles (non fermés pour rénovation)
    dispo = [j for j in range(nb_bur) if j not in indispos_idx.get(p, [])]
    capacite = {j: 2 for j in dispo}

    # ÉTAPE 1 : Placer P et S en priorité (contrainte d'exclusion critique)
    for s in [idx_P, idx_S]:
        restant = eff[s]
        candidats = sorted([(scores[s, j], j) for j in dispo], reverse=True)

        for (_, j) in candidats:
            if restant <= 0:
                break
            if capacite[j] <= 0:
                continue

            # Vérifier l'exclusion P-S : pas de cohabitation
            if s == idx_P and Y[idx_S, j] > 0:
                continue
            if s == idx_S and Y[idx_P, j] > 0:
                continue

            # Vérifier l'exclusion P-S : pas de voisinage
            conflit = False
            for v in voisins[bureaux[j]]:
                vi = b_idx[v]
                if vi in dispo:
                    if s == idx_P and Y[idx_S, vi] > 0:
                        conflit = True
                        break
                    if s == idx_S and Y[idx_P, vi] > 0:
                        conflit = True
                        break
            if conflit:
                continue

            # Assigner le maximum de personnes possible
            assign = min(restant, capacite[j], 2)
            Y[s, j] = assign
            capacite[j] -= assign
            restant -= assign

        # Échec si on n'a pas pu placer tout le service
        if restant > 0:
            return None

    # ÉTAPE 2 : Placer les autres services (O, T, M)
    for s in range(nb_serv):
        if s in [idx_P, idx_S]:
            continue

        restant = eff[s]
        candidats = sorted([(scores[s, j], j) for j in dispo], reverse=True)

        for (_, j) in candidats:
            if restant <= 0:
                break
            if capacite[j] <= 0:
                continue

            assign = min(restant, capacite[j], 2 - Y[s, j])
            Y[s, j] += assign
            capacite[j] -= assign
            restant -= assign

        if restant > 0:
            return None

    return Y


def verifier_solution(Y_all):
    """
    Vérifie qu'une solution complète respecte toutes les contraintes.

    Paramètres:
        Y_all : np.array (nb_serv, nb_phases, nb_bur) - solution complète

    Retourne:
        (bool, str) : (validité, message d'erreur ou "OK")
    """
    for p in range(nb_phases):
        Y = Y_all[:, p, :]

        # Contrainte : bureaux fermés inoccupés
        for j in indispos_idx.get(p, []):
            if np.sum(Y[:, j]) > 0:
                return False, f"Phase {p}: bureau fermé {bureaux[j]} occupé"

        # Contrainte : effectifs corrects
        for s in range(nb_serv):
            if np.sum(Y[s, :]) != eff[s]:
                return False, f"Phase {p}: effectif {services[s]} = {np.sum(Y[s, :])} ≠ {eff[s]}"

        # Contrainte : capacité ≤ 2
        for j in range(nb_bur):
            if np.sum(Y[:, j]) > 2:
                return False, f"Phase {p}: capacité {bureaux[j]} dépassée"

        # Contrainte : exclusion P-S
        for j in range(nb_bur):
            # Cohabitation
            if Y[idx_P, j] > 0 and Y[idx_S, j] > 0:
                return False, f"Phase {p}: cohabitation P-S en {bureaux[j]}"
            # Voisinage
            if Y[idx_P, j] > 0:
                for v in voisins[bureaux[j]]:
                    if Y[idx_S, b_idx[v]] > 0:
                        return False, f"Phase {p}: voisinage P-S"

    # Contrainte : Phase 0 = état initial
    if not np.array_equal(Y_all[:, 0, :], etat_initial):
        return False, "Phase 0 ≠ état initial"

    return True, "OK"


def calculer_mouvements_personnes(Y_all):
    """
    Calcule le nombre total de mouvements en PERSONNES.

    Un mouvement = une personne qui change de bureau entre deux phases.

    Paramètres:
        Y_all : np.array (nb_serv, nb_phases, nb_bur) - solution complète

    Retourne:
        (int, list) : (total, détail par transition)
    """
    total = 0
    details = []

    # Mouvements entre phases consécutives
    for p in range(nb_phases - 1):
        mvt = 0
        for s in range(nb_serv):
            # Compter les départs (diminution d'occupation)
            mvt += np.sum(np.maximum(Y_all[s, p, :] - Y_all[s, p + 1, :], 0))
        details.append(int(mvt))
        total += mvt

    # Mouvements vers l'état final
    mvt_final = 0
    for s in range(nb_serv):
        mvt_final += np.sum(np.maximum(Y_all[s, nb_phases - 1, :] - etat_final[s, :], 0))
    details.append(int(mvt_final))
    total += mvt_final

    return int(total), details


# ----------------------------------------------------------------------------
# Boucle d'arrondi randomisé
# ----------------------------------------------------------------------------

print("\nRecherche de solutions entières...")
nb_trials = 10000
solutions = []
np.random.seed(42)  # Pour reproductibilité
t0 = time.time()

for trial in range(nb_trials):
    if trial % 2000 == 0:
        print(f"  Essai {trial}... ({len(solutions)} solutions)")

    # Tirage aléatoire et projection
    r = np.random.randn(V.shape[1])
    proj = V @ r

    # Normalisation pour cohérence avec u_0 = 1
    if proj[0] < 0:
        proj = -proj

    # Extraction des scores par (service, phase, bureau)
    scores = np.zeros((nb_serv, nb_phases, nb_bur))
    for s in range(nb_serv):
        for p in range(nb_phases):
            for j in range(nb_bur):
                scores[s, p, j] = proj[index(s, p, j)]

    # Ajout de bruit pour diversifier les solutions
    scores += np.random.randn(nb_serv, nb_phases, nb_bur) * 0.3

    # Construction de la solution phase par phase
    Y_all = np.zeros((nb_serv, nb_phases, nb_bur), dtype=int)
    ok = True

    for p in range(nb_phases):
        Y_p = reparer_phase_personnes(scores[:, p, :], p)
        if Y_p is None:
            ok = False
            break
        Y_all[:, p, :] = Y_p

    # Vérification et stockage si valide
    if ok:
        valid, msg = verifier_solution(Y_all)
        if valid:
            mvt, details = calculer_mouvements_personnes(Y_all)
            solutions.append({'Y': Y_all.copy(), 'mvt': mvt, 'details': details})

print(f"\nTemps : {time.time() - t0:.1f}s")


# ============================================================================
# AFFICHAGE DES RÉSULTATS
# ============================================================================

print(f"\n{'─' * 60}")
print(f"Solutions trouvées : {len(solutions)} / {nb_trials}")

if solutions:
    mouvements = [sol['mvt'] for sol in solutions]
    print(f"  Min: {min(mouvements)}, Max: {max(mouvements)}, Moy: {np.mean(mouvements):.1f}")

    # Sélection de la meilleure solution
    best = min(solutions, key=lambda x: x['mvt'])
    Y_best = best['Y']

    print(f"\n{'=' * 60}")
    print(f"MEILLEURE SOLUTION ENTIÈRE : {best['mvt']} MOUVEMENTS")
    print(f"{'=' * 60}")

    # Affichage détaillé de chaque phase
    for p in range(nb_phases):
        print(f"\n{'─' * 60}")
        print(f"PHASE {p} (fermés: {indispos.get(p, [])})")
        print(f"{'─' * 60}")
        afficher_etat(Y_best[:, p, :])

        # Vérification de l'exclusion P-S
        P_burs = [bureaux[j] for j in range(nb_bur) if Y_best[idx_P, p, j] > 0]
        S_burs = [bureaux[j] for j in range(nb_bur) if Y_best[idx_S, p, j] > 0]
        print(f"\n  P dans: {P_burs}")
        print(f"  S dans: {S_burs}")
        print(f"  Exclusion P-S: ✓")

    # Affichage de l'état final cible
    print(f"\n{'─' * 60}")
    print(f"ÉTAT FINAL CIBLE (Figure 2)")
    print(f"{'─' * 60}")
    afficher_etat(etat_final)

    # Détail des mouvements par transition
    print(f"\n{'─' * 60}")
    print("DÉTAIL DES MOUVEMENTS (en personnes)")
    print(f"{'─' * 60}")

    for p in range(nb_phases - 1):
        print(f"\n  Phase {p} -> Phase {p + 1} : {best['details'][p]} personnes")
        for s in range(nb_serv):
            Y_before = Y_best[s, p, :]
            Y_after = Y_best[s, p + 1, :]
            departs = []
            arrivees = []
            for j in range(nb_bur):
                diff = Y_after[j] - Y_before[j]
                if diff < 0:
                    departs.append(f"{bureaux[j]}({-diff})")
                elif diff > 0:
                    arrivees.append(f"{bureaux[j]}(+{diff})")
            if departs or arrivees:
                print(f"    {services[s]}: quitte {departs} -> arrive {arrivees}")

    print(f"\n  Phase 4 -> État Final : {best['details'][-1]} personnes")
    for s in range(nb_serv):
        Y_last = Y_best[s, nb_phases - 1, :]
        Y_cible = etat_final[s, :]
        departs = []
        arrivees = []
        for j in range(nb_bur):
            diff = Y_cible[j] - Y_last[j]
            if diff < 0:
                departs.append(f"{bureaux[j]}({-diff})")
            elif diff > 0:
                arrivees.append(f"{bureaux[j]}(+{diff})")
        if departs or arrivees:
            print(f"    {services[s]}: quitte {departs} -> arrive {arrivees}")

    print(f"\n{'=' * 60}")
    print(f"TOTAL : {best['mvt']} MOUVEMENTS DE PERSONNES")
    print(f"{'=' * 60}")

    # Comparaison et analyse finale
    print(f"\n{'─' * 60}")
    print("COMPARAISON ET ANALYSE")
    print(f"{'─' * 60}")
    print(f"  Borne SDP fractionnaire : {total_frac:.1f} mouvements (en présences)")
    print(f"  Estimation en personnes : {total_frac * 2:.1f} mouvements (×2)")
    print(f"  Solution entière        : {best['mvt']} mouvements")
    print(f"  Gap d'intégrité         : {best['mvt'] - total_frac * 2:.0f} ({100 * (best['mvt'] - total_frac * 2) / max(total_frac * 2, 0.1):.0f}%)")

    print(f"""
  Analyse :
  - La Phase 0 est FIXÉE à l'état initial (Figure 1)
  - L'état final cible est celui de la Figure 2
  - Le SDP modélise des PRÉSENCES binaires (relaxation)
  - La réparation assigne des PERSONNES (Y ∈ {{0,1,2}})

  Conclusion Question 8 :
  - L'arrondi randomisé direct ne donne pas de solutions valides
  - Avec réparation heuristique : {len(solutions)}/{nb_trials} solutions (100%)
  - Solution RÉALISABLE : {best['mvt']} mouvements
  - L'intervalle d'optimalité est [{total_frac * 2:.0f}, {best['mvt']}] en personnes
""")

else:
    print("\n Aucune solution trouvée - vérifier les contraintes")

RELAXATION SDP - PROBLÈME PARIS-DUCHESSE

────────────────────────────────────────────────────────────
ÉTAT INITIAL (Figure 1) - Phase 0
────────────────────────────────────────────────────────────
  A1: [P×2] (2/2)
  A2: [P×2] (2/2)
  B1: [O×1, M×1] (2/2)
  B2: [S×2] (2/2)
  B3: [O×1, T×1] (2/2)
  C1: [T×2] (2/2)
  C2: [S×2] (2/2)
  D1: [T×1] (1/2)
  D2: [M×1] (1/2)
  D3: [M×2] (2/2)
  E1: [vide] (0/2)
  E2: [vide] (0/2)

Vérification effectifs: [4, 4, 2, 4, 4] = [4, 4, 2, 4, 4]

────────────────────────────────────────────────────────────
ÉTAT FINAL CIBLE (Figure 2) - Après Phase 4
────────────────────────────────────────────────────────────
  A1: [P×2] (2/2)
  A2: [P×2] (2/2)
  B1: [M×1] (1/2)
  B2: [M×2] (2/2)
  B3: [M×1] (1/2)
  C1: [S×2] (2/2)
  C2: [S×2] (2/2)
  D1: [T×1] (1/2)
  D2: [T×2] (2/2)
  D3: [T×1] (1/2)
  E1: [O×1] (1/2)
  E2: [O×1] (1/2)

Vérification effectifs: [4, 4, 2, 4, 4] = [4, 4, 2, 4, 4]

QUESTION 6 : FORMULATION SDP

Dimension : N = 300 variables binaires (pr

(CVXPY) Jan 28 11:49:38 PM: Your problem has 90601 variables, 91297 constraints, and 0 parameters.
(CVXPY) Jan 28 11:49:38 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jan 28 11:49:38 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jan 28 11:49:38 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jan 28 11:49:38 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jan 28 11:49:38 PM: Compiling problem (target solver=SCS).
(CVXPY) Jan 28 11:49:38 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Jan 28 11:49:38 PM: Applying reduction Dcp2Cone
(CVXPY) Jan 28 11:49:39 PM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jan 28 11:49:39 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Jan 28 11:50:14 PM: Applying reduction SCS
(CVXPY) Jan 28 11:50:14 PM: Finished problem compilation (took 3.564e+01 seconds).
(CVXPY) Jan 28 11:50:14 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.11 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 45451, constraints m: 46147
cones: 	  z: primal zero / dual free vars: 611
	  l: linear vars: 85
	  s: psd vars: 45451, ssize: 1
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 5000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 47042, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res 

C:\Users\Magomed Tsitsiev\anaconda3\Lib\site-packages\cvxpy\problems\problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
(CVXPY) Jan 28 11:56:55 PM: Problem status: optimal_inaccurate
(CVXPY) Jan 28 11:56:55 PM: Optimal value: -2.468e+02
(CVXPY) Jan 28 11:56:55 PM: Compilation took 3.564e+01 seconds
(CVXPY) Jan 28 11:56:55 PM: Solver (including time spent in interface) took 4.005e+02 seconds


  5000| 5.31e-03  1.10e-04  8.68e-03 -2.47e+02  1.08e+00  4.00e+02 
------------------------------------------------------------------
status:  solved (inaccurate - reached max_iters)
timings: total: 4.00e+02s = setup: 1.31e-01s + solve: 4.00e+02s
	 lin-sys: 1.21e+01s, cones: 3.84e+02s, accel: 8.69e-01s
------------------------------------------------------------------
objective = -246.768811 (inaccurate)
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------

────────────────────────────────────────────────────────────
Status : optimal_inaccurate
Valeur optimale : -246.77
Temps : 436.2s
Rang effectif de U* : 11 / 301

────────────────────────────────────────────────────────────
SOLUTION FRACTIONNAIRE (probabilités de présence)
──────────